In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
criminals_path = '/content/drive/MyDrive/criminals/'

# Check if it exists
if os.path.exists(criminals_path):
    print("✓ Found criminals folder!")
    files = os.listdir(criminals_path)
    print(f"  Contains {len(files)} files")
else:
    print("✗ Folder not found. Creating it...")
    os.makedirs(criminals_path)

Mounted at /content/drive
✓ Found criminals folder!
  Contains 3316 files


In [2]:
!pip install transformers
!pip install faiss-cpu
!pip install faiss-gpu
!pip install -U bitsandbytes
!pip install qwen_vl_utils
!pip install pandas
!pip install  torchvision
!pip install accelerate
!pip install chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 39.6 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 34.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 8.4 

In [3]:
import torch
import sklearn
from torch import nn
from torchvision import transforms
from PIL import Image

In [4]:
import re
def preprocess_text(result):
# Your original text


# Option 1: Get the matched text and convert to lowercase
  matches = re.search(r'assistant\s*\n([\s\S]*)',result, re.IGNORECASE)


  if matches:
    # Group 1 contains the text after "assistant"
    final_answer = matches.group(1).strip().lower()  # .group(1) extracts the captured part
  else:
    final_answer = "none"
  return final_answer


In [5]:
def answer_to_number(results):
  for i in range(len(results)):
     if results[i] == "yes" or results[i] == "yes.":
       results[i] = 1
     elif results[i] == "no" or results[i] == "no.":
       results[i] = 0
     else :
       results[i] = -1
  return results
def computation(labels,results):
  FN,TN,FP,TP,accur = 0,0,0,0,0
  for i in range(len(labels)):
     if labels[i] == 1 and results[i] == 1:
       TP += 1
     elif labels[i] == 1 and results[i] == 0:
       FN += 1
     elif labels[i] == 0 and results[i] == 1:
       FP += 1
     elif labels[i] == 0 and results[i] == 0:
       TN += 1
     else:
       continue
  for i in range(len(labels)):
    if labels[i] == results[i]:
      accur += 1
  accuracy = accur/len(labels)
  LR_PLUS = (TP/(TP+FN))/(FP/(FP+TN))
  LR_MINUS = (FN/(TP+FN))/(TN/(FP+TN))
  NPV = TN/(TN+FN)
  answer = {
      "LR+":LR_PLUS,
      "LR-":LR_MINUS,
      "NPV":NPV,
      "accuracy":accuracy
  }
  return answer
def collection(results):
  combo = {"yes":0,"no":0,"others":0}
  for i in range(len(results)):
    if results[i] == "yes" or results[i] == "yes.":
      combo["yes"] += 1
    elif results[i] == "no" or results[i] == "no.":
      combo["no"] += 1
    else:
      combo["others"] += 1
  return combo

In [6]:
from transformers import BitsAndBytesConfig,AutoModelForImageTextToText,AutoProcessor

model_intern = AutoModelForImageTextToText.from_pretrained(
    "OpenGVLab/InternVL3_5-8B-HF",
    device_map="auto",
    trust_remote_code=True
)
processor_intern = AutoProcessor.from_pretrained(
    "OpenGVLab/InternVL3_5-8B-HF",
    trust_remote_code=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/841 [00:00<?, ?it/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/913 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

video_preprocessor_config.json: 0.00B [00:00, ?B/s]

In [7]:
import pandas as pd
df1 = pd.read_csv('train_offense_facts.csv', on_bad_lines='skip')
df2 = pd.read_csv('test_preprocessed_with_images_and_caste (1).csv', on_bad_lines='skip')
df1 = df1[['id','label','only_facts']]
df2 = df2[['id','label','facts_and_arguments','Caste']]

argument_keywords = [
    'hence',
    'oppose',
    'opposes',
    'opposed',
    'opposing',
    'support',
    'supports',
    'supported',
    'supporting',
    'bailable',
    'granted',
    'rejected'
]

only_facts = []
for fact_arg in df2['facts_and_arguments']:
    sents = fact_arg.split('. ')
    new_sents = []
    for s in sents:
        flag = True
        for key in argument_keywords:
            if key in s:
                flag = False
                break
        if flag:
          new_sents.append(s)
    only_facts.append('. '.join(new_sents))
df2.loc[:, 'only_facts'] = only_facts


In [8]:
revised_caste = ['unknown', 'yadav', 'jat', 'jat', 'gurjar', 'jat', 'musalman', 'rajput', 'meena', 'jat', 'jat sikh', 'muslim', 'rajput', 'dhanka', 'bawri', 'khatik', 'bawri', 'rajput', 'jat', 'bishnoi', 'musalman', 'jatsikh', 'agarwal', 'kahar', 'thakur', 'bavri', 'muslim', 'rajput', 'gurjar', 'sindhi', 'jat', 'nayak', 'rajput', 'khatik', 'jat', 'jat', 'raysikh', 'jat', 'jat', 'jattsikh', 'muslim', 'meena', 'rajpoot', 'sansi', 'jat', 'bishnoi', 'musalman', 'bawaria', 'harijan', 'jat', 'khangar', 'pokharna', 'soni', 'basfod', 'khatik', 'sansi', 'mevati', 'rajput', 'raigar', 'unknown', 'meena', 'musalman', 'sharma', 'jat', 'sindhi', 'jat', 'jat', 'mahajan', 'sansi', 'meena', 'musalman', 'sansi', 'sharma', 'musalman', 'jat', 'gurjar', 'muslim', 'bishnoi', 'muslman', 'bhat', 'unknown', 'jat', 'jatav', 'muslman', 'sansi', 'jat', 'mali', 'sansi', 'nayak', 'balai', 'musalman', 'jaat', 'momdan', 'jatsikh', 'brahaman', 'rajput', 'mdrasi,hindu', 'jat', 'mali', 'muslman', 'saini', 'rajput', 'jat', 'bawari', 'mali', 'thakur', 'saini', 'jat', 'deshwali', 'bishnoi', 'koli', 'khatik', 'meena', 'sen', 'khatik', 'achray (sharma)', 'kumahar', 'rajput', 'jogrash', 'sindhi', 'harijan', 'shah musalman', 'majbi sikh', 'meena', 'pathan', 'rawat', 'daroga', 'momdan', 'khatik', 'sansi', 'muslim', 'aggarwal', 'kayam khani', 'rajput rav', 'nath', 'jat', 'modi', 'rajpoot', 'gurjar', 'sansi', 'maali', 'maali', 'rajput', 'panjabi', 'agarwal', 'jaat', 'nayak', 'meena', 'brahaman', 'meena', 'baori', 'meena', 'muslim', 'jat', 'gurjar', 'harijan', 'nayak', 'muslim', 'bavri', 'khatik', 'nath', 'muslim', 'meena', 'musalman', 'kumawat', 'gosai', 'bawri', 'gupta', 'bishnoi', 'khatik', 'harijan', 'sharma', 'meena', 'sansi', 'arora', 'rajput', 'rajput', 'sansi', 'sansi', 'sansi', 'dakot', 'kayasth', 'gurjar', 'jat', 'saini', 'teli', 'brahmin', 'rajput', 'jat sikh', 'sharma', 'mochi', 'meena', 'ravat', 'jat', 'nayak', 'nayak', 'kharik', 'soni', 'soni', 'sindhi', 'raysikh', 'nayak', 'bangali', 'sansi', 'megwal', 'brahman', 'rajput', 'sad', 'rajput', 'mahajan', 'mogiya', 'agerwal', 'jat', 'jat sikh', 'meghwal', 'gurjar', 'rajput', 'braman', 'musalman', 'jat', 'bhat', 'harizan', 'unknown', 'vyopari', 'jat', 'nath', 'meena', 'kahar', 'bishnoi', 'muslim', 'chhajgirya', 'jat', 'gurjar', 'bhambhi', 'khatik', 'meena', 'sanei', 'barahaman', 'sindhi', 'rajput', 'jat', 'brahman', 'aggarwal', 'jat(bhadu)', 'jat', 'musalman', 'jat', 'gurjar', 'sansi', 'meena', 'jat', 'mogya', 'majbisikh', 'khatik', 'meena', 'mali', 'khichi musalman', 'rajput', 'kanjar', 'rajput', 'sansi', 'sansi', 'gurjar', 'unknown', 'jat', 'rajput', 'rajput', 'gujjar', 'jat', 'jat', 'muslim', 'muslim', 'bishnoi', 'musalman', 'jat', 'soni', 'meena', 'jatsikh', 'mogya', 'mali', 'rajput', 'khatik', 'nayak', 'harijan', 'rajpoot', 'walmiki', 'meena', 'rajaput', 'jat', 'khatik', 'brahmin', 'kanjar', 'muslaman', 'sansi', 'jat sikh', 'muslim', 'khatik', 'mali', 'nayak', 'mali', 'mev', 'bishnoi', 'rebari', 'rebari', 'jat', 'jat', 'bishnoi', 'muslim', 'meghwal', 'meena', 'brahaman', 'unknown', 'balai', 'dhanka', 'meena', 'jat', 'rajput', 'gurjar', 'berwa', 'musalman', 'meena', 'jat', 'koli', 'tiwari', 'meghwal', 'chobdar muslman', 'brahman', 'chhipa', 'jat', 'parjapat', 'rana', 'jat', 'kaymkhani', 'dhobi', 'majbisikh', 'muslim', 'jatav', 'panjra muslman', 'kaji', 'sindhi', 'meena', 'mali', 'jat', 'kaymkhani', 'gujar', 'soni', 'sansi', 'meena', 'sansi', 'muslman luhar', 'muslim', 'kanjar', 'brahaman', 'yadav', 'sunar', 'meena', 'balai', 'muslman', 'muslman', 'jatav', 'mehrasikh', 'bishnoi', 'agarwal', 'sindhi', 'charan', 'muslim', 'sharma', 'meena', 'unknown', 'rajpoot', 'ragir', 'kumawat', 'jat', 'rajpoot', 'raisikh', 'jat', 'gujar', 'bawariya', 'sadh', 'nayak', 'muslima', 'bawari', 'meena', 'meena', 'nayak', 'unknown', 'jat', 'bishnoi', 'musalman', 'kayamkhani muslaman', 'muslim', 'mali', 'sindhi', 'rajpoot', 'bawari', 'musalman', 'rajput', 'mev', 'musalman', 'gurjar', 'mali', 'walmiki', 'musalman', 'mali', 'jat', 'babaji', 'meena', 'musalman', 'jat', 'gujer', 'jat', 'sanshi', 'nayak', 'rajput', 'jat sikh', 'lodha', 'brahman', 'brahmin', 'sharma', 'muslim', 'muslaman', 'mali', 'jatav', 'daroga', 'muslman', 'musalman', 'shansi', 'unknown', 'muslim', 'jat', 'sindhi', 'gupta', 'muslim', 'rawat', 'jat', 'swami', 'jat', 'mali', 'rajpoot', 'swami', 'rawat', 'rajput', 'gurjar', 'raigar', 'muslman', 'meena', 'daroga', 'jat', 'khatik', 'musalman', 'rangrej', 'dhoby', 'sawami', 'meena', 'valmiki', 'musalman', 'balai', 'valmiki', 'raisikh', 'valmiki', 'raisikh', 'raisikh', 'ramdasia', 'mali', 'musalman', 'mehara', 'jat', 'musalman kasai', 'muslman', 'bishnoi', 'ramdasiya', 'natha', 'gurjar', 'bishnoi', 'muslim', 'bishnoi', 'khatik', 'khatik', 'rajput', 'rajaput', 'koli', 'bhargav', 'sunar', 'musalman', 'muslim', 'nagori muslman', 'sindhi', 'jat', 'meena', 'harijan', 'jatav', 'muslman', 'meo', 'agarwal', 'rajput', 'bishanoi', 'rajput', 'sindhi', 'gurjar', 'jat', 'bishnoi', 'sindhi', 'bhand', 'jat', 'soni', 'musalman', 'nayak', 'jat', 'brahmin', 'raiger', 'muslman', 'gurjar', 'rajput', 'mahajan', 'muslman', 'musalman', 'harijan', 'jat', 'meghwal', 'mali', 'hindu', 'unknown', 'sanshi', 'daroga', 'jataw', 'sansi', 'bishnoi', 'bavri', 'harijan', 'harijan', 'yadav', 'muslman', 'jat', 'raisikh', 'bavri', 'sindhi', 'rai sikh', 'meena', 'rajpoot', 'jat', 'jat', 'muslim / khan', 'gurjar', 'rajput', 'kumawat', 'mogya', 'rajput', 'jat', 'musalman', 'sharma', 'nayak', 'jat', 'musalman', 'jat', 'soni', 'bishnoi', 'nayak', 'jat', 'musilm', 'meena', 'jat', 'meena', 'muslim', 'musalman', 'balai', 'jat', 'kayamkhani', 'gurjar', 'jain', 'nayak', 'bawari', 'sansi', 'jat', 'jat', 'sansi', 'meena   [ s.t ]', 'muslim', 'rai sikh', 'rajput', 'sai', 'raigar', 'musalman', 'musalman', 'sharma', 'jat', 'bharawa', 'gurjar', 'harijan', 'jat', 'rajput', 'rajput', 'musalman', 'muslman', 'muslman', 'bishnoi', 'muslim', 'rajput', 'rajput', 'sargara', 'muslman', 'rajput', 'unknown', 'boari', 'rajput', 'meena', 'panjabi', 'dholi', 'jat', 'rager', 'sansi', 'nath', 'jat', 'bangali', 'gurjar', 'bhutta muslman', 'deswali musalman', 'chajgriya', 'nayak', 'nayak', 'musalman', 'gurjar', 'jat', 'unknown', 'meghwal', 'jaat', 'meena', 'muslman', 'sansi', 'bawari', 'jat', 'sansi', 'rajput', 'meena', 'soni', 'jat', 'panjabi', 'sansi', 'musalman', 'khati,hindu', 'bisayati', 'muslim', 'gurjar', 'bhanwariya jat', 'bawari', 'yadav', 'arora', 'khatik', 'ray sikh', 'jat', 'chita', 'rawat', 'gurjar', 'bhati muslim', 'unknown', 'jangir ( khati', 'jangir ( khati', 'jat', 'khati', 'pathan', 'mogya', 'jat', 'yadav', 'musalman', 'banjara', 'jaat', 'jat', 'meena', 'valmiki', 'valmiki', 'muslman', 'musalman', 'musalman (teli)', 'sansi', 'parjapat', 'rao', 'khtik', 'musalman', 'unknown', 'unknown', 'sindhi', 'muslman', 'gurjar', 'chouhan', 'raisikh', 'gurjar', 'rajput', 'jat', 'jat', 'jat', 'luhar', 'jat', 'sansi', 'jat', 'harijan', 'bairwa', 'meghwanshi', 'meena', 'sansi', 'shindhi', 'jat', 'khatik', 'unknown', 'harizan', 'gujrar', 'punjabi', 'mahajan', 'rajpoot', 'jaat', 'meena', 'raika', 'rawat', 'mali', 'arora', 'jaat', 'kayamkhani', 'jat', 'rajput', 'mali', 'unknown', 'harijan', 'meena', 'sansi', 'rawat', 'ood', 'jat', 'bishnoi', 'rajpoot', 'bishnoi', 'jat', 'musalman', 'jat', 'saini', 'jat', 'meena', 'khatik', 'majahbi sikh', 'muslim', 'mehra', 'sindhi', 'muslman', 'jat', 'rawat', 'lodha', 'rawat', 'jat', 'brahman', 'sansai', 'meena', 'rajpoot', 'lodha', 'muslim', 'jaat', 'meghwal', 'regar', 'arora', 'musalman', 'pinara muslman', 'musalman', 'jat', 'meena', 'sindhi', 'gahlot', 'nayak', 'pinara musalman', 'muslim', 'muslim', 'jat', 'muslman', 'gurjar', 'rajput', 'raika', 'meena', 'jat', 'gurjar', 'muslim', 'choudhary(muslim)', 'sunar', 'muslim', 'jat', 'sharma', 'musalman', 'meena', 'musalman', 'sansi', 'muslman', 'gurjar', 'jat', 'jat', 'sansi', 'jat', 'sansi', 'mehrat', 'rajput', 'kumbhar', 'mali', 'jat', 'jat', 'prajapat', 'deshwali', 'rajpoot', 'mali', 'bansal', 'gurjar', 'mathur', 'jogi', 'meena', 'yadav', 'muslman', 'ravna rajput', 'ravna rajput', 'meena', 'besayti muslman', 'parjapat', 'rawat', 'gurjar', 'muslim', 'meghwal', 'bavri', 'bishnoi', 'bishnoi', 'jat', 'rajpoot', 'ray sikh', 'meena', 'kayast', 'goswami', 'bishnoi', 'musalman', 'kalal', 'muslim', 'jat', 'maali', 'rajput', 'gurjar', 'kayasth', 'khatik', 'brahman', 'sasi', 'sharma', 'harijan', 'rawat', 'unknown', 'musalmaan', 'rajput', 'nayak', 'meena', 'dhobi', 'jat', 'gurjar', 'mev', 'jat', 'khateek', 'yadav', 'musalman', 'bishnoi (bhanwal)', 'jat', 'bavari', 'gurjar', 'jaat', 'meena', 'meena', 'jat', 'rawal bhat', 'sansi', 'jaat', 'rajpoot', 'unknown', 'bheel', 'muslman', 'nayak', 'meena', 'rajpoot', 'ray sikha', 'jat', 'valmiki', 'rav', 'brahaman', 'sansi', 'jat', 'deswali musalman', 'meena', 'sharma', 'musalman', 'shrma', 'aroda', 'raysikh', 'gurjar', 'muslim', 'jat', 'kayamkhani', 'musalman', 'khatik', 'meena', 'majhbisikh', 'muslim', 'rajput', 'muslman', 'muslim', 'mhajan', 'mali', 'rao', 'kumhar', 'meena', 'hindu', 'muslim', 'sansi', 'niyargar', 'muslim', 'meena', 'unknown', 'muslim', 'balai', 'rajput', 'gurjar', 'somvanshi', 'jat', 'bishnoi', 'rajput', 'kalal', 'kumhar', 'meghawal', 'meena', 'rawat', 'gujar', 'muslim', 'meena', 'brahaman', 'meghwal', 'sansi', 'gurjar', 'gurjar', 'rajput', 'panjabi', 'dhanka', 'rawat', 'agarwal', 'nayak', 'jat', 'muslim', 'vaishnav', 'mali', 'bawari', 'jat', 'jat', 'gurjar', 'musalmn', 'meena', 'bishnoi', 'bishnoi', 'chita', 'jat', 'sikari rajput', 'jat', 'kumhar', 'brahman', 'sansi', 'musalman', 'sharma', 'bhambhi', 'rajput', 'rajput', 'meena', 'chippa', 'gurjar', 'rajput', 'meena', 'jat sikh', 'giwariya', 'musalman', 'agrwal', 'kumhar', 'teli', 'rajput', 'panjabi', 'teli muslman', 'jat', 'dhobi', 'muslim', 'mogya', 'gurjar', 'musalman', 'jatsikh', 'kaymkhani', 'dhanak', 'jat', 'jaat', 'rajpoot', 'sansi', 'vishnoi', 'kahar,hindu', 'sidh', 'saiyad musalman', 'arora', 'musalman', 'dhanak', 'muslim', 'arora', 'visnoi', 'meena', 'rajpoot', 'mehart', 'nath', 'nayak', 'jat', 'sharma', 'gurjar', 'sharma', 'sansi', 'sindhi', 'mansuri muslman', 'kahar', 'thakur', 'jat', 'harijan', 'meo', 'meghwal', 'kathat', 'rajput', 'mali', 'shindhi', 'vishnoi', 'bairwa', 'panjabi khatri', 'soni', 'rajput', 'khateek', 'jat', 'valmiki', 'brahman', 'jat', 'sansi', 'sansi', 'meena', 'jaat', 'jat', 'gurjar', 'rajput', 'sbc', 'meena', 'meena', 'rajput', 'gurjar', 'rajpoot', 'muslaman', 'muslaman', 'jat', 'khatik', 'jat', 'muslim', 'unknown', 'bagariya', 'rajput', 'meo', 'sad', 'sad', 'rajput', 'koli', 'dhkar', 'musalman', 'valmiki', 'jat', 'fakir musalan', 'jat', 'thakur', 'jat', 'rawat', 'meena', 'kalal', 'gurjar', 'jat', 'gurjar', 'rawat', 'gurjar', 'balai', 'musaman', 'musalman', 'ode', 'rajput', 'jat', 'meghwal', 'jat', 'kushwah', 'muslman', 'gurjar', 'balai', 'jat', 'gurjar', 'pathan musalman', 'muslim', 'rajput', 'jain', 'odd', 'regar', 'gori muslman', 'jat', 'jat', 'sindhi', 'rajput', 'gurjar', 'muslim', 'unknown', 'jaat', 'bishnoi', 'luhar', 'sharma', 'harijan', 'gosawami', 'muslman', 'jat', 'mev', 'musalman', 'rajput', 'meghwal', 'harijan', 'bishnoi', 'sansi', 'luhar', 'mev', 'sindhi', 'unknown', 'meena', 'meghwal', 'deswali', 'yadav', 'jat', 'musalman', 'mali', 'sharma', 'arora', 'parjapati', 'rajpoot', 'mahajan', 'sansi', 'rajput', 'bisnoi', 'mali', 'bawri', 'harijan', 'jatav', 'raysikh', 'rajput', 'mali', 'kumhar', 'meena', 'nayak', 'muslim', 'kumawat', 'muslim', 'meena', 'jat', 'jat', 'pandit', 'gurjar', 'jat', 'bawari', 'gurjar', 'jat', 'rajput', 'sakka', 'jangid', 'rajput', 'meena', 'koli', 'ansari musalman', 'rajput', 'bishnoi', 'bishnoi', 'rajput', 'chhipa', 'jat', 'jat', 'sindhi', 'brahman', 'sai muslman', 'jat', 'musalman', 'gurjar', 'meghwal', 'meena', 'jat', 'agarwal', 'rajput', 'unknown', 'panjabi', 'harijan', 'jat', 'walmiki', 'balai', 'meena', 'jat', 'khtik', 'soni', 'balmik', 'muslman', 'meena', 'jat', 'rajjput', 'mali', 'panjabi', 'rajput', 'sharma', 'unknown', 'jat sikh', 'muslman', 'meena', 'bairwa', 'pathan', 'sunar', 'shindi', 'jat', 'khatik', 'jat', 'bishnoi', 'arora', 'gurjar', 'muslman', 'sain', 'jat', 'meena', 'sharma', 'jat', 'brahmin', 'unknown', 'musalman', 'rajput', 'rajput', 'jat', 'musalman', 'rajput', 'meena', 'musalman', 'gurjar', 'rajput', 'nayak', 'sindhi', 'muslman', 'nayak', 'sansi', 'unknown', 'kayamkhani', 'gurjar', 'jat', 'brahmin', 'khitik', 'jatav', 'mehrat', 'nat', 'jat', 'jat', 'mali', 'jat', 'jat', 'meena', 'pathan musalman', 'harijan', 'brahman', 'muhslman', 'mushlim', 'muslim', 'meena', 'musalman', 'gawariya', 'meena', 'rajput', 'unknown', 'meena', 'nayak', 'meena', 'mali', 'mehrat', 'gurjar', 'meena', 'vaishnav', 'brahaman', 'hariyana brahaman', 'jat', 'sindhi', 'musalman', 'jat', 'jat', 'jat', 'rajput', 'meena', 'nayak', 'unknown', 'mali', 'muslim', 'muslim', 'luhar', 'luhar', 'luhar', 'meena', 'harijan', 'harijan', 'meena', 'jat', 'shrama', 'khatik', 'ray sikh', 'raisikh', 'thakur', 'kaymkani', 'sc', 'khatik', 'droga', 'muslim', 'nayak', 'khatik', 'unknown', 'jat', 'kayamkhani musalman', 'kayamkhani musalman', 'mali', 'jat', 'brahman', 'bishnoi', 'jat', 'jat', 'gurjar', 'soni', 'brahmin', 'jatav', 'bavri', 'meena', 'kayamkhani', 'khatik', 'muslim', 'muslman', 'gurjar', 'nayak', 'gujer', 'mali', 'sharma', 'vaishnav', 'bharman', 'chita', 'jat sikh', 'unknown', 'ravana rajput', 'chajgreya', 'bawariya', 'balai', 'jat', 'brahmin', 'dhanka', 'kumhar (prajapat)', 'sansi', 'unknown', 'soni', 'mev', 'khatik', 'gurjar', 'jat', 'musalman', 'quareshi musalman', 'raisikh', 'mehara', 'muslman', 'meena', 'harijan', 'meena', 'sansi', 'jat', 'jat', 'jat', 'kalbeliya', 'musalman', 'sindhi', 'teli  musalman', 'jat', 'bagra', 'nayak', 'aacharua', 'meghawal', 'charan', 'patwa', 'jatsikh', 'vijayvargiya', 'chipa', 'meena', 'muslim', 'muslim', 'oad', 'keer', 'muslim', 'muslmaan', 'mahajan', 'daroga', 'mev', 'gurjar', 'agarwal', 'cheeta', 'rajput', 'meena', 'sansi', 'gurjar', 'muslman', 'teli', 'sardar', 'bawri', 'bawri', 'khatik', 'jat', 'chrishan', 'bawaryi', 'bawaryi', 'muslman', 'sansi', 'deswali musalman', 'gurjar', 'rajput', 'meena', 'muslamaan', 'bawri', 'meena', 'rai sikh', 'bairwa', 'unknown', 'majabi', 'unknown', 'khatik', 'sansi', 'gujjar', 'ravna rajput', 'rajpoot', 'jat', 'jat', 'mali', 'gurjar', 'bishnoi', 'baraman', 'jat', 'jat', 'nai', 'jat', 'walmiki', 'jat', 'arora', 'raysikh', 'unknown', 'muslman', 'mali', 'sikari', 'sharma', 'kohli', 'gujar', 'agarwal', 'sasi', 'sindhi', 'muslim', 'gurjar', 'kumawat', 'jat', 'bawaryi', 'bawaryi', 'gurjar', 'bishnoi', 'jat', 'rajput', 'muslim', 'rajput', 'muslim', 'mali', 'jat sikh', 'khatik', 'rajput', 'lakhara', 'bagra brahimin', 'brahman', 'musalman', 'meghwal', 'mali', 'muslman', 'jatsikh', 'jat', 'chobdar', 'bavri', 'gurjar', 'harijan', 'arora', 'gurjar', 'sansi', 'unknown', 'sansi', 'babriya', 'harijan', 'meena', 'charan', 'muslim', 'khateek', 'rajpoot', 'khatik', 'unknown', 'sansi', 'rajput', 'mahawat', 'gurjar', 'brahmin', 'sindhi', 'luhar', 'bishanoi', 'mali', 'muslim', 'klal', 'klal', 'rajput', 'musalman', 'bhishti musalman', 'prajapat', 'bavriya', 'sindhi', 'jat', 'muslim', 'muslim', 'meena', 'rajput', 'musalman', 'jat', 'meena', 'muslman', 'regar', 'kumhar', 'nayak', 'meena', 'musalman', 'rajput', 'muslim', 'jat', 'jat', 'mehrat', 'raigar', 'mali', 'meena', 'sharma', 'jat', 'swami', 'brahman', 'babariya', 'rajput', 'musalman', 'rajpoot', 'bisayati', 'meena', 'musalman', 'jaat', 'pathan musalman', 'jat', 'unknown', 'jaat', 'gurjar', 'rajput', 'teli musalman', 'shani', 'kalal', 'gurjer', 'meena', 'jat', 'jatsikh', 'dakot', 'kumawat', 'jat', 'brahaman', 'musalman', 'jat', 'khetek', 'sansi', 'jat', 'unknown', 'mali', 'charan', 'meena', 'rawat', 'rajpoot', 'musalman', 'ojha', 'sansi', 'kumawat', 'rajpoot', 'muslim', 'khatik', 'muslaman', 'meena', 'moslim', 'muslim', 'sunar', 'teli', 'rajput', 'jat', 'jatav', 'meo', 'rajput', 'musalman', 'sansi', 'jat', 'jat', 'jat', 'sansi', 'mali', 'jat', 'gurjar', 'fakir musalman', 'meena', 'prjapt', 'gurjar', 'musalman', 'pathan muslman', 'meena', 'nayak', 'mali', 'rajput', 'mali', 'jat', 'gupta', 'jat', 'mhajan', 'mev', 'muslim', 'nai', 'raigar', 'musalman', 'jat', 'daroga', 'muslim', 'musalman', 'goswami', 'meena', 'rajput', 'jat', 'kanjar', 'mahajan', 'jat', 'ansari muslim', 'aachariya', 'jat', 'musalman', 'kashmirisikh', 'rajput', 'majbi', 'jat', 'nayak', 'jatsikh', 'bharman', 'meena', 'unknown', 'meena', 'mahajan', 'rajput', 'soni', 'muslim', 'teli  (musalman)', 'meena', 'meena', 'musalman', 'rajput', 'jat', 'rajput', 'rajput', 'bairwa', 'gurjar', 'sansi', 'rajput', 'rawat', 'muslim', 'shrivastav', 'meghwal', 'jat', 'jat', 'jaat kasniya', 'jat', 'jat', 'jat', 'braman', 'meena', 'meena', 'jat', 'muslim', 'sansi', 'jat', 'bishnoi', 'jat', 'khatik', 'daroga', 'meena', 'rajput', 'rajput', 'sasi', 'rajput', 'sindhi', 'sindhi', 'meena', 'rawat', 'kumawat', 'muslim', 'yadav', 'mushalman', 'jat', 'gurjar', 'soni', 'yadav', 'patwa', 'ray sikh', 'nath', 'bishnoi', 'unknown', 'gurjar', 'mali', 'jat', 'meena', 'gurjar', 'jat', 'raysikh', 'rawat', 'jat', 'meena', 'bhambhi', 'jat', 'jat', 'gurjar', 'parjapat', 'jat', 'mahajan', 'harijan', 'rjapoot', 'raysikh', 'agrwal', 'brahman', 'agarwal', 'jat', 'muslim', 'mali', 'sansi', 'gavariya', 'meena', 'muslim', 'gurjar', 'mehra', 'jat', 'sahu teli', 'gurjar', 'kanjar', 'swami', 'rajput', 'gurjar', 'khati', 'chipa', 'muslman', 'sansi', 'sansi', 'rajput', 'jat', 'musalman', 'gurjar', 'muslim', 'rawat', 'musalman', 'harijan', 'mogya', 'nayak', 'teli musalman', 'jat', 'koli', 'muslman', 'jat', 'bishnoi', 'regar', 'jat', 'meena', 'rajput', 'meena', 'meena', 'jat', 'rajput', 'meena', 'bishnoi', 'bishnoi', 'gurjar', 'musalman', 'meena', 'gurjar', 'modi', 'kaymkahni', 'bishnoi', 'meena', 'meena', 'jat', 'bhambi (maru)', 'musalman', 'kanjar', 'kushwah', 'muslim', 'chhipa muslman', 'megwal', 'meena', 'brahimin', 'unknown', 'kanjar', 'bishnoi', 'jat', 'jat', 'bisnoi', 'jat', 'kanjar', 'soni', 'luhar', 'dhanka', 'meena', 'meena', 'gurjar', 'sindhi', 'jat', 'rajput', 'raigar', 'rajput', 'dholi', 'sipahi musalman', 'raisikh', 'dholee', 'ray sikh', 'kaymkhani', 'rajput', 'jat', 'gurjar', 'yadav', 'vijayvargiya', 'rajput', 'muslman', 'muslman', 'rajput', 'bheel', 'brahamin', 'gurjar', 'musaman', 'sasi', 'gurjar', 'harijan', 'musalman', 'panjabi', 'meena', 'sen', 'rajput', 'khati', 'sansi', 'raisikh', 'yogi', 'jat', 'raisikh', 'meena', 'meena', 'rajput', 'meena', 'jaat', 'nayak', 'meena', 'mev', 'jat', 'muslim', 'meena', 'musalman', 'muslim', 'meena', 'jat', 'mochi bheel', 'jat', 'goswami', 'muslim', 'charan', 'kanjar', 'gurjar', 'brahman', 'rajput', 'jat', 'jat', 'saansi', 'saansi', 'bavriya', 'sasi', 'shekh muslman', 'gujjar', 'musalman', 'dhank', 'muslim', 'muslim', 'rajput', 'bavri', 'meena', 'jat', 'chajgariya', 'musalman', 'unknown', 'muslim teli', 'balai', 'charan', 'jat', 'brahman', 'bhambhi', 'gurjar', 'koli', 'bramhin', 'gurjar', 'musalman', 'meghwal', 'teli musalman', 'jat', 'bishnoi', 'raisikh', 'harijan', 'jat', 'jat', 'kumawat', 'jat', 'bhakhar', 'bishnoi', 'jat', 'arora', 'rajput', 'bavri', 'meena', 'bhambhi', 'sharma', 'khitk', 'meena', 'meena', 'jat', 'musalman', 'muslim', 'mansuri/ pinara', 'shansi', 'kumawat', 'gurjar', 'parjapat', 'jatsikh', 'muslim', 'meo', 'rajpoot', 'meena', 'bheel', 'shrama', 'jat', 'musalman', 'vishnoi', 'jat', 'sidh', 'mali', 'mehara', 'rajput', 'sindhi', 'bairwa', 'meena', 'jaat', 'panjabi', 'meena', 'rajpoot', 'bagda brahmin', 'swami', 'pathan muslim', 'saini', 'meena', 'bhargav', 'dhanak', 'sansi', 'brahaman', 'meena', 'muslman', 'gurjar', 'sargra', 'mhajan', 'nai', 'bavri', 'meena', 'muslim', 'barahman', 'musalman', 'jatiya', 'swami', 'meena', 'thakur', 'mehra sikh', 'gesawat musalman', 'jatsikh', 'gurjar', 'baghela', 'tank', 'unknown', 'sinhdi', 'sigiwal', 'aroda', 'mirasi musalman', 'jat', 'rana', 'sharma', 'bajigar', 'rajput', 'jat', 'rajput', 'jat', 'gurjar', 'sansi', 'jatshikh', 'gurjar', 'sharma', 'jat', 'muslman', 'rajpoot', 'bawria', 'jat', 'jat', 'sansi', 'mewara', 'swami', 'unknown', 'muslman', 'mirasi', 'gujair', 'rajput', 'aroda', 'unknown', 'jat', 'meena', 'jat', 'chudigar muslman', 'meena', 'bawari', 'bavriya', 'nai', 'jat', 'sharma', 'harijan', 'harijan', 'jat', 'jat', 'jat sikha', 'mali', 'mev', 'jat', 'shekh muslman', 'mali', 'unknown', 'sipahi musalman', 'meena', 'rajpoot', 'yadav', 'kanjar', 'gurjar', 'jaat', 'koda', 'muslim', 'rawat', 'yadav', 'saini (mali)', 'gurjar', 'brahman', 'rajput', 'musalman', 'sindhi', 'luhar', 'musalman', 'jat', 'harijan', 'muslim', 'dholi', 'pujari', 'sindhi', 'soni', 'meena', 'muslman', 'meena', 'kayamkhani', 'jat', 'jat', 'jat', 'sansi', 'jat', 'shindi', 'vapari', 'bavri', 'muslman', 'agrwal', 'meena', 'jat', 'gurjar', 'kumavat', 'muslim', 'mogya', 'sansi', 'jat', 'sansi', 'muslman banjara', 'musalman', 'muslim', 'musalman', 'kanjar', 'unknown', 'nayak', 'daroga', 'majbisikh', 'musalman', 'dhanka', 'nayak', 'kasai', 'swami', 'valmiki', 'jat', 'rajput', 'unknown', 'panjabi', 'harijan', 'jat', 'kumawat', 'bramin', 'rajput', 'jat', 'rajput', 'muslim', 'mahawahar', 'gurjar', 'jat', 'jat', 'meena', 'gancha', 'mushalman', 'dhakad', 'raigar', 'raigar', 'muslim', 'unknown', 'jatav', 'brahaman', 'sindhi', 'ray sikh', 'jat', 'meena', 'raisikh', 'jaat', 'jat', 'arora', 'rajpoot', 'muslim', 'muslman', 'meena', 'muslman', 'baraman', 'baraman', 'rajput', 'musalmman', 'jat', 'jain', 'rajput', 'harijan', 'rajput', 'brahman', 'muslman', 'yadav', 'dakot', 'muslman', 'soni', 'gurjar', 'nayak', 'jain', 'muslim', 'sindhi', 'musalman', 'jat', 'brahmin', 'gurjar', 'muslim', 'brahamman', 'sansi', 'gurjar', 'jatsikh', 'sindhi', 'gurjar', 'meena', 'mali', 'mali', 'jat', 'mali', 'raysikh', 'bawri', 'meena', 'jat', 'unknown', 'rajput', 'silawat', 'gurjar', 'meghwal', 'bawariya', 'harizan', 'soni', 'sharma', 'bishnoi', 'khati', 'musalman', 'meena', 'mali', 'jaat', 'kasmiri brahman', 'rajput', 'jat', 'rajpoot', 'rajpuroit', 'sansi', 'sindhi', 'gurjar', 'majbi sikh', 'rawat', 'musalman', 'kumhar', 'muslim', 'meena', 'jat', 'meena', 'meena', 'muslim', 'raisikh', 'jat', 'janwar', 'yadav', 'nayak', 'mogya', 'raisikh', 'jat', 'gurjar', 'kayamkhani', 'jat', 'muslim', 'rawat', 'gurjar', 'mali', 'banjara muslim', 'meena', 'unknown', 'meena', 'sindhi', 'mali', 'babaji', 'sindhi', 'shansi', 'thekur', 'sindhi', 'mev', 'mali', 'jaat', 'maheswri', 'yadav', 'sansi', 'rawat', 'musalman', 'gurjar', 'rajpoot', 'dhobi', 'mehra', 'jat', 'harijan', 'harijan', 'soni', 'bhisti muslman', 'rajput', 'jat', 'jat', 'muslman', 'muslim', 'harijan', 'mev', 'gurjar', 'mali', 'gurjar', 'bairwa', 'raysikh', 'mali', 'unknown', 'muslim', 'raigar', 'deswali muslman', 'yadav', 'barhmana', 'muslim', 'jat', 'kumhaar', 'muslim', 'meghwal', 'jat', 'meghwal', 'muslman', 'bishnoi', 'jat', 'saini', 'gurjar', 'jaat', 'rangrej mohmdan', 'sahu', 'rajput', 'pathan muslim', 'sunar', 'khateek', 'khatik', 'obc', 'ghelot', 'mochi', 'shah', 'vishnoi', 'goswami', 'arora', 'unknown', 'musalman', 'jat', 'sansi', 'jat', 'rawat', 'rawat', 'kanjar', 'jatsikh', 'jat', 'mali', 'meena', 'mushalman', 'muslim', 'rajput', 'maheswari', 'musalman', 'jat', 'rajpoot', 'jaat', 'jain', 'rajpurohit', 'mehrat', 'soni', 'sharma', 'paswan', 'balmik', 'parjapat', 'sharma', 'sindhi', 'meena', 'nayak', 'mev', 'patwa', 'bawari', 'brahaman', 'jat', 'muslim', 'jat', 'bavri', 'luhar', 'jat', 'jat', 'dhadhi muslaman', 'gurjar', 'khatik', 'bavariya', 'sindhi', 'deswali', 'jat', 'kaymkhani', 'sriwastaw', 'bairwa', 'kumhar', 'arora', 'musalman', 'kanjar', 'rajput', 'jat', 'mali', 'arora', 'vaishnav', 'meena', 'sindhi', 'mali', 'meena', 'parasar', 'parjapat', 'rajput', 'unknown', 'meghwal', 'mali', 'charan', 'meena', 'gurjar', 'panjabi', 'mali', 'mogya', 'valmike', 'jat', 'gurjar', 'rajput', 'muslim', 'musalman', 'oad', 'musalman bhisty', 'kanjar', 'sansi', 'sansi', 'jat', 'jat', 'dakot pandit', 'nayak', 'chhajgariya', 'koli', 'mali', 'suthar', 'muslim', 'nayak', 'jat', 'jat', 'meena', 'jat', 'aroda', 'sindhi', 'musalman', 'mev', 'jat', 'jat', "bhat musalman, 'sikligar'", 'kanjar', 'kayamkhani', 'nayak', 'bavariya', 'jat', 'jat', 'chita', 'gurjar', 'bhargaw', 'vaisnav', 'jat', 'reger', 'kumhar prajapat', 'unknown', 'somvansi', 'sansi', 'meena', 'musalman', 'bavari', 'jat', 'lohar', 'chamar', 'meena', 'meena', 'meena', 'rajput', 'bhati (muslim)', 'meghwal', 'meena', 'koli', 'sindhi', 'sindhi', 'meena', 'meena', 'muslim', 'brahmin', 'muslmnan', 'sawami', 'vijaywergia', 'unknown', 'yadav', 'soni', 'joshi', 'raika(dewasi)', 'mali', 'jat', 'bishnoi', 'muslim', 'bhambhi', 'patawa', 'jangir', 'brahman', 'mev', 'koli', 'naai', 'musalman', 'musalman', 'kasai mushalman', 'bagra brahmin', 'meena', 'unknown', 'meena', 'jat', 'jat', 'rawat', 'choudhary', 'nayak', 'sansi', 'meena', 'nayak', 'musalman', 'sansi', 'gurjar', 'jat', 'mali', 'mali', 'muslim', 'sansi', 'mushalman', 'musalman', 'khatik', 'jat', 'musalman', 'rawat', 'gadiya luhar', 'bawari', 'meena', 'gurjar', 'soni', 'jat', 'gurjar', 'arora', 'rajput', 'muslim', 'brhamin', 'gupta', 'meena', 'meena', 'luhar musalman', 'shikari rajpoot', 'lodha', 'kayamkhani', 'jat', 'meena', 'oda', 'khatik', 'rajput', 'rajput', 'nai', 'unknown', 'unknown', 'nai', 'thapa', 'unknown', 'brahman', 'unknown', 'jat', 'muslman', 'mahawar', 'ond rajput', 'yadav', 'brahman', 'rai sikh', 'unknown', 'kayamkhani', 'dhobi', 'meena', 'mehra', 'muslman', 'agarwal', 'musalman', 'sikh', 'unknown', 'meo', 'brahman', 'rajput', 'rajput', 'chita', 'meena', 'garg', 'sansi', 'bavariya', 'rajpoot', 'mev', 'shindhi', 'hussian', 'unknown', 'muslman', 'unknown', 'khatik', 'jat', 'harijan', '(walmiki) harijan', 'meena', 'majbi sikh', 'jatsikha', 'khatik', 'sunar', 'jat', 'mushalman', 'gurjar', 'gurjar', 'meena', 'sansi', 'meena', 'meena', 'choudhary', 'khatik', 'rajput', 'sharma', 'jat', 'muslim', 'gurjar', 'meena', 'musalman', 'suwalka', 'meena', 'jat', 'rajput', 'bavari', 'muslim', 'khatri', 'mogya', 'kasai', 'regar', 'muslim', 'jatsikh', 'mushalman', 'gujer', 'jatsikh', 'raigar', 'brahaman', 'raigar', 'gurjar', 'daroga', 'jat', 'jat', 'meena', 'meena', 'muslman', 'babaria', 'rajpoot', 'mahajan', 'bavri', 'kaymkani muslman', 'muslman banjara', 'bairawa', 'bawaryi', 'bawaryi', 'raysikh', 'raisikh', 'mev', 'aroda', 'jat', 'meena', 'tak', 'meena', 'muslim', 'khatik', "kanjar (sc) hindu", 'meena', 'mehrat', 'arora', 'bishnoi', 'brahaman', 'meena', 'meena', 'jat', 'harijan', 'rajput', 'kayamkhani', 'khatik', 'panjabi', 'jat', 'thakur', 'jat', 'musalman', 'jat', 'mev', 'meena', 'meena', 'brahman', 'swami', 'swami', 'gurjar', 'jat', 'khatik', 'gaddi musalman', 'pandit', 'jat', 'jat', 'sadh', 'khatik', 'khatik', 'jatsikh', 'bishnoi', 'jat', 'thakur', 'muslman', 'musalman', 'koli', 'ramgdhiya', 'brahman', 'rajpoot', 'mev', 'thakur', 'jat', 'khati', 'jat', 'gurjar', 'kumawat', 'meena', 'harijan', 'gurjar', 'brahman', 'gurjar', 'bishnoi', 'bishnoi', 'kumhar', 'musalman', 'dakot', 'rajput', 'vaishnav ramawat', 'musalman', 'khatik', 'rajput', 'muslim', 'unknown', 'khatik', 'jat', 'panjabi', 'rajput', 'chipa', 'mali', 'banbagria', 'meena', 'prajapat', 'kumawat', 'pathan muslim', 'rajpoot', 'muslman', 'saini', 'daroga', 'jat', 'meena', 'raiger', 'sharma', 'gurjar', 'musalman', 'kanjar', 'vaishay', 'meena', 'nayak', 'bawri', 'meena', 'bishnoi', 'arora (sindhi)', 'gurjar', 'muslim', 'meena', 'jaat', 'brahman', 'rawat', 'musalman banjara', 'unknown', 'meena', 'brahmin', 'jattsikh', 'jat', 'koli', 'gurjar', 'rajpoot', 'unknown', 'jatav', 'mali', 'unknown', 'brahman', 'daroga', 'sindhi', 'kanjar', 'meena', 'muslaman', 'dhakar', 'gurjer', 'kumhar', 'jat', 'tailor', 'jat', 'sunar', 'unknown', 'musalman', 'raisikh', 'sansi', 'meena', 'gurjar', 'dhobi', 'meena', 'nayak', 'muslim', 'jatav', 'gurjar', 'musalman', 'rajput', 'mali', 'musalman', 'bavari', 'gurjar', 'meghwal', 'meena', 'jat', 'rajput', 'mali', 'koli', 'gurjar', 'kumahar', 'gurjar', 'jat', 'musalman', 'jat sikh', 'esai', 'gurjar', 'raisikh', 'muslim', 'meena', 'musalman', 'jat', 'rajput', 'mev', 'dhanka', 'jat', 'raisikh', 'bavriya', 'musalman', 'arora', 'muslim', 'pthan', 'musalman', 'nayak', 'sansi', 'raigar', 'jat sikh', 'bishnoi', 'balai', 'jat', 'bheel', 'arora', 'rajpoot', 'mali', 'gurjar', 'jat', 'sanshi', 'meena', 'jattsikh', 'kheldar musalman', 'khatik', 'arora', 'dhanak', 'thakur', 'jat', 'jat', 'muslim', 'meena', 'muslim', 'musalman  teli', 'mali', 'rajput', 'dhakad', 'rajpoot', 'bawri', 'jat', 'gurjar', 'meena', 'unknown', 'jat', 'bawari', 'musalman', 'kumawat', 'muslman', 'musalman', 'unknown', 'musalman', 'bishnoi', 'sansi', 'kalal', 'gurjar', 'brahmman', 'raisikh', 'rebari', 'chhajgariya', 'rajput', 'meena', 'meena', 'jat', 'gurjar', 'nayak', 'rajput', 'meena', 'jat', 'regar', 'gujar', 'jat', 'meena', 'jat', 'pathan', 'musalman', 'meena', 'rajput', 'rajput', 'teli', 'rajpoot', 'jat', 'jatav', 'bairva', 'gurjar', 'unknown', 'balai', 'muslim', 'jatsikh', 'brahman', 'musalman', 'meena', 'maghwal', 'majbee sikh', 'shikari', 'musalman', 'jat', 'mewafaros', 'gurjar', 'gurjar', 'gurjar', 'meena', 'rajpoot', 'gurjar', 'jaiswal', 'muslim', 'jat', 'musalman', 'solanki', 'meena', 'gurjar', 'ganral', 'musalman', 'meena', 'jangir', 'meena', 'gurjar', 'muslim', 'jat', 'jat', 'brahmin', 'bhraman', 'mandal bihari', 'jat', 'bhat', 'gurjar', 'musalman', 'meena', 'gurjar', 'bairwa', 'ganral', 'harijan', 'rajput', 'muslim', 'meghwal', 'meghwal', 'muslim', 'musalman', 'jat (tandi)', 'jat', 'muslim', 'bishnoi', 'meena', 'meena', 'mev', 'gurjar', 'jat', 'bishnoi', 'bishnoi', 'muslim', 'saiya, muslman', 'raisikh', 'gurjar', 'deshvali muslman', 'meena', 'nath', 'nayak', 'jat', 'kayamkhani muslim', 'meena', 'joge', 'rajput', 'nayak', 'bishnoi', 'rajput', 'rajput', 'sindhi', 'musalman', 'unknown', 'mev', 'harijan', 'rajput', 'bengali', 'jaat', 'jat', 'musalman', 'gurjar', 'kumawat', 'shekh musalman', 'chohan(muslman)', 'mali', 'jaat godara', 'muslim', 'somvanshi', 'jat', 'musalman', 'sindhi', 'jatsikh', 'mali', 'gurjar', 'jat', 'gurjar', 'baweri', 'baweri', 'jat', 'muslman', 'ansari musalman', 'meena', 'tamboli', 'gurjar', 'sansi', 'muslim', 'rawat', 'rajput', 'meghwal', 'mali', 'sunar', 'muslim', 'khatik', 'majahabi sikh', 'muslima', 'bihari', 'sindhi', 'brahaman', 'musalman', 'rajput', 'seni', 'jat', 'rajput', 'jat', 'rajput', 'ganral', 'somvanshi', 'meena', 'raysikh', 'vaishnav', 'musalman deshwali', 'walmiki', 'unknown', 'meena', 'unknown', 'unknown', 'dhanak', 'muslim', 'gusai', 'meena', 'mev', 'muslim', 'sindhi', 'musalman', 'gurjar', 'muslim', 'jogi', 'koli', 'rajput', 'swami', 'rajput', 'nayak', 'jat', 'muslman', 'rajput', 'rajput', 'vyapyari musalman', 'musalman', 'unknown', 'jat', 'bawariya', 'agarwal', 'gurjar', 'meena', 'meena', 'sharma', 'brihaman', 'muslman', 'jat', 'gurjar', 'rajput', 'chita', 'jat', 'kaymkhani', 'jaat   bhambhu', 'bangali', 'gurjar', 'pathan', 'mali', 'bishnoi', 'meena', 'muslman', 'rawat', 'sansi', 'jat', 'jat', 'jat', 'balai', 'shikari', 'nayak', 'meghwal', 'gurjar', 'rajput', 'deswali muslman', 'jaat', 'jat', 'jat', 'gurjar', 'bishnoi', 'pathan muslaman', 'brahman', 'mallaha', 'jat', 'jat', 'meo', 'meena', 'meena', 'unknown', 'kumawat', 'kumawat', 'musalman', 'rai sikh', 'kasara', 'bishnoi', 'jat', 'majbi sikh', 'sindhi', 'rajput', 'damami muslman', 'rajput', 'bawari', 'jat', 'mathur', 'ramghariya', 'gurjar', 'dheemar', 'musalman', 'nath', 'kumawat', 'jat', 'meena', 'bishnoi', 'jat', 'mev', 'meghwal', 'meena', 'unknown', 'majbi sikh', 'khatik', 'musalman', 'meena', 'jat', 'odrajput', 'rawat', 'bawari', 'swami', 'meo', 'kabra', 'mev', 'nath', 'meena', 'jaat', 'rajput', 'jat', 'gurjar', 'nayak', 'luhar', 'meena', 'jat', 'khatik', 'christian', 'rajpoot', 'jatsikh', 'jat (manda)', 'sansi', 'musalman', 'khatik', 'jatshik', 'jat', 'lavana sikh', 'jat(chaudhary)', 'khatik', 'panjabi', 'kashmirisikh', 'musalman', 'rajput', 'brahmin', 'musalman', 'panjabi sikh', 'meena', 'jat', 'kanjar', 'musalaman', 'rajput', 'bagairay', 'musalman', 'meena', 'mev', 'mena', 'raysikh', 'majbi sikh', 'muslim', 'khatik', 'jat', 'gurjar', 'jatav', 'gurjar', 'hariyana brahamn', 'jat', 'rajpur', 'mali', 'sansi', 'rajput', 'nagori musalman', 'meena', 'kanjar', 'koli', 'mahajan', 'bawari', 'jangid', 'nayak', 'unknown', 'musalman', 'swami', 'brahmano ki sareri', 'teli', 'sikh', 'mus,', 'sikhh', 'nayak', 'meena', 'meena', 'gurjar', 'valmiki', 'kamboj sikh', 'yadav', 'rai sikh', 'thakur', 'majbisikh', 'jat', 'majahbi sikh', 'ramgadhia', 'jatsikh', 'rai sikh', 'oad rajpoot', 'jat', 'jat', 'thakur', 'jat', 'sharma', 'kanjar', 'sansi', 'mahajan', 'nayak', 'rajput', 'bramin', 'bishnoi', 'bishnoi', 'jat sikh', 'charan', 'vishnoi', 'gurjar', 'nayak', 'gurjar', 'rajpoot', 'gurjar', 'gurjar', 'unknown', 'muslmaan', 'kayamkhani', 'sharma', 'gurjar', 'musalman', 'gunsai', 'jangir', 'rajput', 'meena', 'kashmirisikh', 'unknown', 'sorger', 'kheldar muslaman', 'musalman', 'bawari', 'bishnoi', 'saini', 'qureshi muslman', 'muslim', 'jat', 'rajput', 'jain', 'nayak', 'rajput', 'jatsikh', 'deswali musalman', 'meena', 'harijan', 'gurjar', 'meena', 'rajput', 'jat', 'meena', 'jat', 'bishanoi', 'mev', 'gurjar', 'jat sikh', 'brahman', 'sen', 'rawat', 'mewati muslman', 'raisikh', 'unknown', 'banjara', 'nayak', 'kanjar', 'muslman', 'muslim', 'musalman', 'musalman', 'bishnoi', 'gurjar', 'bavri', 'musalman', 'jat', 'jatav', 'sone', 'jogi', 'meena']

In [9]:
for i in (range(len(revised_caste))):
  df2.loc[i,'Caste'] = revised_caste[i]

In [10]:
print(df2['Caste'])

0       unknown
1         yadav
2           jat
3           jat
4        gurjar
         ...   
3311      jatav
3312       sone
3313       jogi
3314      meena
3315      MEENA
Name: Caste, Length: 3316, dtype: object


In [11]:
print(df2)

                                             id  label  \
0      Bail Application_2180_202002-01-20211157      0   
1       Bail Application_1017_202006-07-2020391      1   
2      Bail Application_1156_202122-02-20215574      1   
3     Bail Application_101049_202131-03-2021293      1   
4      Bail Application_4458_202006-10-20202515      1   
...                                         ...    ...   
3311  Bail Application__1545_202112-03-20211846      1   
3312           Bail Appl__4218_201920-12-201970      0   
3313    Bail Application_750_202105-03-20211151      0   
3314    Bail Application_584_202102-02-20212940      0   
3315     Bail Application_321_202017-02-2020527      1   

                                    facts_and_arguments    Caste  \
0     When the plaintiff Kibahan told the above thin...  unknown   
1     According to the prosecution, the inspector-in...    yadav   
2     The accused is in judicial custody. The learne...      jat   
3     The investigator has comp

In [18]:
results2 = []
labels = [df2['label'].iloc[i] for i in range(len(df2))]
for i in range(len(df2)):
 test_img_path = f"/content/drive/MyDrive/criminals/{i:05d}.jpg"
 image = Image.open(test_img_path)
 image = image.resize((224, 224), Image.BICUBIC)
 image = image.convert("RGB")
 text = df2['only_facts'].iloc[i]
 label = df2['label'].iloc[i]
 caste = df2['Caste'].iloc[i]
 system_prompt = '''Instruction: You are provided with the facts of a criminal case report.
                    You need to understand how a bail decision is made from the facts in order to respond to the question.'''
 user_prompt = f'''Question: The facts of the current case are the following: {text}
                  You are also given an image of the accused person which is the photo of the accused which you need to see.
                  The accused person who belongs to {caste} caste.
                  Should the accused person be granted bail? Please answer in yes/no only.
                  \n Answer:'''

 conversation = [
     {
         "role": "system",
         "content": system_prompt
     },
    {
        "role": "user",
        "content": [
            {"type": "image", "image" : image},
            {"type": "text", "text": user_prompt}
        ]
    }
]
 prompt = processor_intern.apply_chat_template(conversation, add_generation_prompt=True)
 inputs = processor_intern(images=image, text=prompt, return_tensors="pt")
 inputs = inputs.to("cuda")
 generated_output = model_intern.generate(**inputs, return_dict_in_generate=True,
                                         output_scores=True,
                                         do_sample=True,
                                         max_new_tokens=256,
                                         temperature=0.1)

 # Extracting the generated text from the output of the model
 answer_text = processor_intern.decode(generated_output.sequences[0], skip_special_tokens=True)

 # The original prompt includes the "Answer:" prefix, so we need to remove it from the generated text
 # Find the position of the last "Answer:" and take the substring after it.
 answer_start_index = answer_text.rfind("Answer:")
 if answer_start_index != -1:
     answer_text = answer_text[answer_start_index + len("Answer:"):].strip()
 else:
     answer_text = answer_text.strip()

 print(i+1)

 ans = preprocess_text(answer_text)
 print(ans)
 results2.append(ans)

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


1
yes
2
no


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


3
yes
4
no


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


...

3314
no


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


3315
yes


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


3316
yes


In [19]:
print(results2)

['yes', 'no', 'yes', 'no', 'no', 'yes', 'yes', 'no', 'yes', 'yes.', 'yes', 'no', 'no', 'no', 'no', 'yes', 'yes', 'no', 'no', 'no', 'no', 'no', 'yes', 'no', 'no', 'yes', 'no', 'no', 'yes', 'yes', 'no', 'yes', 'yes', 'no', 'no', 'yes', 'yes', 'yes', 'yes', 'no', 'yes', 'no', 'yes', 'yes', 'yes', 'no', 'no', 'yes', 'yes', 'yes', 'yes', 'yes', 'no', 'no', 'yes', 'no', 'yes', 'yes', 'yes', 'no', 'yes.', 'no', 'no', 'no', 'yes', 'no', 'yes', 'no', 'no', 'yes', 'yes', 'yes', 'yes', 'yes.', 'yes.', 'yes', 'no', 'yes', 'no', 'no', 'yes.', 'yes', 'yes', 'no', 'yes', 'no', 'no', 'yes', 'no', 'yes', 'yes', 'yes', 'yes', 'no', 'yes', 'no', 'no', 'yes', 'yes', 'no', 'yes', 'yes', 'no', 'yes', 'no', 'no', 'yes', 'no', 'no', 'yes', 'yes', 'yes', 'no', 'yes', 'no', 'no', 'no', 'no', 'yes', 'no', 'yes', 'yes', 'yes.', 'yes', 'no', 'yes', 'yes', 'no', 'yes', 'yes', 'yes', 'no', 'no', 'yes', 'no', 'no', 'yes', 'yes', 'no', 'no', 'yes', 'no', 'yes', 'yes', 'no', 'yes', 'no', 'yes', 'no', 'yes', 'yes', 'no'

In [20]:
for i in range(len(results2)):
  matches = re.search(r'\b(yes|no)\b', results2[i], re.IGNORECASE)

  if matches:
    results2[i] = matches.group(1).lower()
  else:
    results2[i] = "none"
print(results2)

['yes', 'no', 'yes', 'no', 'no', 'yes', 'yes', 'no', 'yes', 'yes', 'yes', 'no', 'no', 'no', 'no', 'yes', 'yes', 'no', 'no', 'no', 'no', 'no', 'yes', 'no', 'no', 'yes', 'no', 'no', 'yes', 'yes', 'no', 'yes', 'yes', 'no', 'no', 'yes', 'yes', 'yes', 'yes', 'no', 'yes', 'no', 'yes', 'yes', 'yes', 'no', 'no', 'yes', 'yes', 'yes', 'yes', 'yes', 'no', 'no', 'yes', 'no', 'yes', 'yes', 'yes', 'no', 'yes', 'no', 'no', 'no', 'yes', 'no', 'yes', 'no', 'no', 'yes', 'yes', 'yes', 'yes', 'yes', 'yes', 'yes', 'no', 'yes', 'no', 'no', 'yes', 'yes', 'yes', 'no', 'yes', 'no', 'no', 'yes', 'no', 'yes', 'yes', 'yes', 'yes', 'no', 'yes', 'no', 'no', 'yes', 'yes', 'no', 'yes', 'yes', 'no', 'yes', 'no', 'no', 'yes', 'no', 'no', 'yes', 'yes', 'yes', 'no', 'yes', 'no', 'no', 'no', 'no', 'yes', 'no', 'yes', 'yes', 'yes', 'yes', 'no', 'yes', 'yes', 'no', 'yes', 'yes', 'yes', 'no', 'no', 'yes', 'no', 'no', 'yes', 'yes', 'no', 'no', 'yes', 'no', 'yes', 'yes', 'no', 'yes', 'no', 'yes', 'no', 'yes', 'yes', 'no', 'no'

In [21]:


print("Without RAG:")
print()
print(collection(results2))
results2 = answer_to_number(results2)
print(labels)
print(results2)
print(computation(labels,results2))

Without RAG:

{'yes': 1764, 'no': 1552, 'others': 0}
[np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0

In [22]:
!pip install sentence_transformers
!pip install rank_bm25

In [23]:
import chromadb
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import numpy as np
client = chromadb.Client()
collection = client.create_collection(name="docs", get_or_create=True)

embedder = SentenceTransformer("all-MiniLM-L6-v2").cuda()
docs = [
    df1['only_facts'].iloc[i]  for i in range(len(df1))
]

embeddings = embedder.encode(docs).tolist()

# Split data into smaller batches to avoid exceeding ChromaDB's batch size limit
batch_size = 5000 # Using 5000, which is less than the max_batch_size of 5461
for i in range(0, len(docs), batch_size):
    batch_docs = docs[i:i + batch_size]
    batch_embeddings = embeddings[i:i + batch_size]
    batch_ids = [f"{j}" for j in range(i, min(i + batch_size, len(docs)))]

    collection.add(
        documents=batch_docs,
        embeddings=batch_embeddings,
        ids=batch_ids
    )

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [24]:
def create_message(index,combined_results,combined_labels):
  return f'''case{index+1}:{combined_results[index]}\n
             Based on the facts the final decision of the case is that the bail was {'granted' if combined_labels[index]==1 else 'rejected'}.'''

In [25]:
results1 = []
labels = [df2['label'].iloc[i] for i in range(len(df2))]
for i in range(len(df2)):
  test_img_path = f"/content/drive/MyDrive/criminals/{i:05d}.jpg"
  image = Image.open(test_img_path)
  image = image.resize((224, 224), Image.BICUBIC)
  image = image.convert("RGB")
  results_chroma = collection.query(
    query_embeddings=embedder.encode(df2['only_facts'].iloc[i]).tolist(),
    n_results=3,
    include=["documents", "distances", "metadatas"]
)

  documents = results_chroma['documents'][0]
  sim_score = results_chroma['distances'][0]
  index=results_chroma['ids'][0]
  index = [int(index[i]) for i in range(len(index))]

  combined_labels = [df1['label'].iloc[i] for i in index]

  text = df2['only_facts'].iloc[i]
  label = df2['label'].iloc[i]
  caste = df2['Caste'].iloc[i]
  user_prompt = '''You are a bail decision assistant. Study these example cases and apply the pattern to the new case.

EXAMPLE CASES (learn from these):

'''

# Add examples in CLEAR pattern
  for j in range(len(documents)):
    user_prompt += f'''Case {j+1}:
Facts: {documents[j]}
Decision: {"GRANT BAIL (yes)" if combined_labels[j] == 1 else "DENY BAIL (no)"}

'''

  user_prompt += f'''NEW CASE TO DECIDE:
Facts: {text}
Image: image of the accused
The accused belong to {caste} caste.
INSTRUCTIONS:
1. Compare this new case with the 3 example cases above
2. Look for SIMILAR factors between cases
3. If the new case is similar to majority of the example cases where bail was GRANTED, answer "yes"
4. If the new case is similar to majority of the example cases where bail was DENIED, answer "no"

Important: You MUST base your decision on the comparison with examples.

Decision (yes/no only): '''
  conversation = [

    {
        "role": "user",
        "content": [
            {"type": "image", "image" : image},
            {"type": "text", "text": user_prompt}
        ]
    }
]
  prompt = processor_intern.apply_chat_template(conversation, add_generation_prompt=True)
  inputs = processor_intern(images=image, text=prompt, return_tensors="pt")
  inputs = inputs.to("cuda")
  generated_output = model_intern.generate(**inputs, return_dict_in_generate=True,
                                         output_scores=True,
                                         do_sample=True,
                                         max_new_tokens=256,
                                         temperature=0.1)

 # Extracting the generated text from the output of the model
  answer_text = processor_intern.decode(generated_output.sequences[0], skip_special_tokens=True)

 # The original prompt includes the "Answer:" prefix, so we need to remove it from the generated text
 # Find the position of the last "Answer:" and take the substring after it.
  answer_start_index = answer_text.rfind("Answer:")
  if answer_start_index != -1:
     answer_text = answer_text[answer_start_index + len("Answer:"):].strip()
  else:
     answer_text = answer_text.strip()

  print(i+1)
  ans = preprocess_text(answer_text)
  print(ans)
  results1.append(ans)


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


1
yes


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


2
no
3
yes


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


...

3313
no


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


3314
no


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


3315
yes
3316
yes


In [26]:
print(results1)

['yes', 'no', 'yes', 'no', 'yes', 'no', 'yes', 'yes', 'yes', 'yes', 'no', 'no', 'no', 'no', 'yes', 'yes', 'no', 'no', 'no', 'yes', 'no', 'no', 'yes', 'no', 'yes', 'yes', 'no', 'no', 'yes', 'yes', 'no', 'yes', 'yes', 'no', 'yes', 'yes', 'no', 'no', 'yes', 'yes', 'yes', 'no', 'yes', 'yes', 'yes', 'yes', 'no', 'yes', 'yes', 'yes', 'yes', 'no', 'no', 'no', 'no', 'no', 'yes', 'yes', 'yes', 'yes', 'no', 'yes', 'yes', 'yes', 'yes', 'no', 'yes', 'no', 'no', 'no', 'no', 'yes', 'no', 'no', 'yes', 'yes', 'no', 'yes', 'no', 'no', 'yes', 'no', 'yes', 'yes', 'no', 'yes', 'no', 'yes', 'no', 'yes', 'yes', 'yes', 'yes', 'yes', 'no', 'yes', 'no', 'yes', 'yes', 'no', 'no', 'yes', 'no', 'no', 'yes', 'no', 'yes', 'no', 'no', 'no', 'yes', 'yes', 'yes', 'no', 'no', 'yes', 'no', 'no', 'no', 'no', 'no', 'no', 'yes', 'yes', 'no', 'yes', 'yes', 'yes', 'yes', 'yes', 'yes', 'no', 'no', 'no', 'yes', 'no', 'yes', 'yes', 'no', 'no', 'no', 'no', 'no', 'yes', 'yes', 'yes', 'no', 'yes', 'no', 'yes', 'yes', 'no', 'no', '

In [27]:
def answer_to_number(results):
  for i in range(len(results)):
     if results[i] == "yes" or results[i] == "yes.":
       results[i] = 1
     elif results[i] == "no" or results[i] == "no.":
       results[i] = 0
     else :
       results[i] = -1
  return results
def computation(labels,results):
  FN,TN,FP,TP,accur = 0,0,0,0,0
  for i in range(len(labels)):
     if labels[i] == 1 and results[i] == 1:
       TP += 1
     elif labels[i] == 1 and results[i] == 0:
       FN += 1
     elif labels[i] == 0 and results[i] == 1:
       FP += 1
     elif labels[i] == 0 and results[i] == 0:
       TN += 1
     else:
       continue
  for i in range(len(labels)):
    if labels[i] == results[i]:
      accur += 1
  accuracy = accur/len(labels)
  LR_PLUS = (TP/(TP+FN))/(FP/(FP+TN))
  LR_MINUS = (FN/(TP+FN))/(TN/(FP+TN))
  NPV = TN/(TN+FN)
  answer = {
      "LR+":LR_PLUS,
      "LR-":LR_MINUS,
      "NPV":NPV,
      "accuracy":accuracy
  }
  return answer
def collection(results):
  combo = {"yes":0,"no":0,"others":0}
  for i in range(len(results)):
    if results[i] == "yes" or results[i] == "yes.":
      combo["yes"] += 1
    elif results[i] == "no" or results[i] == "no.":
      combo["no"] += 1
    else:
      combo["others"] += 1
  return combo

In [28]:
def processor(results):
 for i in range(len(results)):
   matches = re.findall(r'\b(yes|no)\b', results[i], flags=re.IGNORECASE)
   results[i] = matches[-1].lower() if matches else "None"
 return results

In [29]:


results1 = processor(results1) #2 nd order preprocessing
print("With RAG:")
print(collection(results1))
results1 = answer_to_number(results1)
print(labels)
print(results1)
print(computation(labels,results1))

With RAG:
{'yes': 1666, 'no': 1650, 'others': 0}
[np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), n